In [1]:
%pip install psycopg2-binary

  Obtaining dependency information for psycopg2-binary from https://files.pythonhosted.org/packages/40/2c/dd379facaa4bd41d7b04711ff30931ca62981c00a91502250ffb49da08a7/psycopg2_binary-2.9.13-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB 217.9 kB/s eta 0:00:13
    --------------------------------------- 0.0/2.8 MB 281.8 kB/s eta 0:00:10
    --------------------------------------- 0.1/2.8 MB 365.7 kB/s eta 0:00:08
   - -------------------------------------- 0.1/2.8 MB 438.1 kB/s eta 0:00:07
   - -------------------------------------- 0.1/2.8 MB 420.8 kB/s eta 0:00:07
   -- ------------------------------------- 0.1/2.8 MB 473.7 kB/s eta 0:00:06
   -- ------------------------------------- 0.2/2.8 MB 482.7 kB/s eta 0:00:06
   --- ------------------------------------ 0.2/2.8 MB 627.5 kB/s eta 0:00:05
   ---- -----


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import pandas as pd
import psycopg2

db_url = os.getenv("DATABASE_URL")
assert db_url, "Falta DATABASE_URL en el entorno"

query_resumen = """
SELECT
    COUNT(*) AS total_filas,
    MIN(ts) AS fecha_minima,
    MAX(ts) AS fecha_maxima,
    COUNT(DISTINCT svc) AS servicios_distintos,
    COUNT(DISTINCT status_code) AS codigos_distintos
FROM public.logs;
"""

query_distribucion = """
SELECT
    svc,
    http_method,
    status_code,
    COUNT(*) AS total_logs,
    ROUND(AVG(duration)::numeric, 2) AS duracion_promedio,
    MAX(duration) AS duracion_maxima
FROM public.logs
GROUP BY svc, http_method, status_code
ORDER BY total_logs DESC;
"""

with psycopg2.connect(db_url) as conn:
  df_resumen = pd.read_sql_query(query_resumen, conn)
  df_dist = pd.read_sql_query(query_distribucion, conn)

display(df_resumen)
display(df_dist)


C:\Users\jesus\AppData\Local\Temp\ipykernel_12672\3295399967.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_resumen = pd.read_sql_query(query_resumen, conn)
C:\Users\jesus\AppData\Local\Temp\ipykernel_12672\3295399967.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dist = pd.read_sql_query(query_distribucion, conn)


,total_filas,fecha_minima,fecha_maxima,servicios_distintos,codigos_distintos
0,35217,2026-11-12 20:36:02.327519+00:00,2026-11-21 01:31:05.695111+00:00,5,10


,svc,http_method,status_code,total_logs,duracion_promedio,duracion_maxima
0,catalog,GET,200,6170,402.12,800
1,auth,POST,201,3174,408.21,800
2,checkout,POST,201,3151,400.52,800
3,users,GET,200,3114,411.93,800
4,payments,POST,201,3060,406.54,799
...,...,...,...,...,...,...
67,payments,POST,502,67,2542.12,4889
68,catalog,PATCH,502,63,2778.73,5000
69,auth,POST,502,62,2777.13,4968
70,catalog,PATCH,503,62,2577.50,4958


In [7]:
query_status = """
SELECT
    status_code,
    COUNT(*) AS total_logs,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_total,
    ROUND(AVG(duration)::numeric, 2) AS duracion_promedio,
    MIN(duration) AS duracion_min,
    MAX(duration) AS duracion_max
FROM public.logs
GROUP BY status_code
ORDER BY status_code;
"""

with psycopg2.connect(db_url) as conn:
  df_status = pd.read_sql_query(query_status, conn)

df_status


C:\Users\jesus\AppData\Local\Temp\ipykernel_12672\2089773659.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_status = pd.read_sql_query(query_status, conn)


,status_code,total_logs,pct_total,duracion_promedio,duracion_min,duracion_max
0,200,17195,48.83,405.45,10,800
1,201,9385,26.65,405.08,10,800
2,400,1081,3.07,416.41,12,800
3,401,1365,3.88,412.35,10,800
4,403,1228,3.49,402.92,10,799
5,404,1452,4.12,399.13,10,800
6,500,1449,4.11,2713.24,402,4999
7,502,613,1.74,2701.90,407,5000
8,503,800,2.27,2743.84,411,4999
9,504,649,1.84,2617.31,411,4999


In [8]:
query_svc = """
SELECT
    svc,
    COUNT(*) AS total_logs,
    COUNT(*) FILTER (WHERE status_code >= 400) AS errores_4xx_5xx,
    COUNT(*) FILTER (WHERE status_code >= 500) AS errores_5xx,
    ROUND(100.0 * COUNT(*) FILTER (WHERE status_code >= 500) / COUNT(*), 2) AS pct_5xx,
    ROUND(AVG(duration)::numeric, 2) AS duracion_promedio
FROM public.logs
GROUP BY svc
ORDER BY pct_5xx DESC, total_logs DESC;
"""

query_diario = """
SELECT
    DATE(ts AT TIME ZONE 'UTC') AS dia_utc,
    COUNT(*) AS total_logs,
    COUNT(*) FILTER (WHERE status_code >= 500) AS errores_5xx,
    ROUND(AVG(duration)::numeric, 2) AS duracion_promedio
FROM public.logs
GROUP BY dia_utc
ORDER BY dia_utc;
"""

with psycopg2.connect(db_url) as conn:
  df_svc = pd.read_sql_query(query_svc, conn)
  df_diario = pd.read_sql_query(query_diario, conn)

display(df_svc)
display(df_diario)

C:\Users\jesus\AppData\Local\Temp\ipykernel_12672\3151273892.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_svc = pd.read_sql_query(query_svc, conn)
C:\Users\jesus\AppData\Local\Temp\ipykernel_12672\3151273892.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_diario = pd.read_sql_query(query_diario, conn)


,svc,total_logs,errores_4xx_5xx,errores_5xx,pct_5xx,duracion_promedio
0,users,7433,1851,760,10.22,643.95
1,catalog,11794,2927,1173,9.95,632.95
2,checkout,7795,1898,775,9.94,631.20
3,payments,4034,974,400,9.92,628.82
4,auth,4161,987,403,9.69,632.99


,dia_utc,total_logs,errores_5xx,duracion_promedio
0,2026-11-12,1031,93,603.91
1,2026-11-13,3824,322,607.80
2,2026-11-14,2070,216,658.14
3,2026-11-15,2987,294,621.59
4,2026-11-16,9743,984,631.89
5,2026-11-17,4454,449,634.95
6,2026-11-18,4231,427,641.92
7,2026-11-19,4032,416,646.21
8,2026-11-20,2194,233,650.57
9,2026-11-21,651,77,680.22


In [9]:
query_horas = """
SELECT
    DATE_TRUNC('hour', ts AT TIME ZONE 'UTC') AS hora_utc,
    COUNT(*) AS total_logs,
    COUNT(*) FILTER (WHERE status_code >= 500) AS errores_5xx,
    ROUND(AVG(duration)::numeric, 2) AS duracion_promedio
FROM public.logs
WHERE DATE(ts AT TIME ZONE 'UTC') = DATE '2026-11-16'
GROUP BY hora_utc
ORDER BY hora_utc;
"""

with psycopg2.connect(db_url) as conn:
  df_horas = pd.read_sql_query(query_horas, conn)

df_horas

C:\Users\jesus\AppData\Local\Temp\ipykernel_12672\2331194914.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_horas = pd.read_sql_query(query_horas, conn)


,hora_utc,total_logs,errores_5xx,duracion_promedio
0,2026-11-16 00:00:00,394,40,623.19
1,2026-11-16 01:00:00,452,36,562.41
2,2026-11-16 02:00:00,420,47,645.63
3,2026-11-16 03:00:00,444,49,651.59
4,2026-11-16 04:00:00,439,58,677.88
5,2026-11-16 05:00:00,421,46,689.48
6,2026-11-16 06:00:00,437,49,710.41
7,2026-11-16 07:00:00,445,40,624.66
8,2026-11-16 08:00:00,442,55,639.11
9,2026-11-16 09:00:00,443,46,636.83


In [1]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()
conn = psycopg2.connect(os.getenv("DATABASE_URL"))

df_muestra = pd.read_sql("SELECT * FROM public.logs ORDER BY ts DESC LIMIT 10;", conn)
print(df_muestra.info())
print(df_muestra.head())

conn.close()

C:\Users\jesus\AppData\Local\Temp\ipykernel_14828\794986360.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_muestra = pd.read_sql("SELECT * FROM public.logs ORDER BY ts DESC LIMIT 10;", conn)


<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   id           10 non-null     int64              
 1   ts           10 non-null     datetime64[us, UTC]
 2   ingested_at  10 non-null     datetime64[us, UTC]
 3   svc          10 non-null     str                
 4   url          10 non-null     str                
 5   http_method  10 non-null     str                
 6   status_code  10 non-null     int64              
 7   log          10 non-null     str                
 8   duration     10 non-null     int64              
dtypes: datetime64[us, UTC](2), int64(3), str(4)
memory usage: 852.0 bytes
None
      id                               ts                      ingested_at  \
0  43951 2026-11-24 18:53:16.241782+00:00 2026-11-24 18:53:16.783631+00:00   
1  43950 2026-11-24 18:53:08.772710+00:00 2026-11-24 18:53:09.513695+00:00   
2 

In [3]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()

query = """
    SELECT
        svc,
        COUNT(*) AS total_logs,
        ROUND(AVG(duration)::numeric, 2) AS avg_duration_ms
    FROM public.logs
    WHERE svc IS NOT NULL
    GROUP BY svc
    ORDER BY total_logs DESC;
"""

with psycopg2.connect(os.getenv("DATABASE_URL")) as conn:
    df_svc = pd.read_sql_query(query, conn)

print(df_svc)

C:\Users\jesus\AppData\Local\Temp\ipykernel_21512\360347199.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_svc = pd.read_sql_query(query, conn)


        svc  total_logs  avg_duration_ms
0   catalog       14692           632.81
1  checkout        9767           636.38
2     users        9366           642.18
3      auth        5101           636.83
4  payments        5025           617.60
